### Rag Pipeline - Data ingestion to Vector DB Pipeline

In [4]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\ACER\AppData\Local\Temp\ipykernel_15464\4064311150.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader


In [5]:
### Read all the pdf from directory
def process_all_pdfs(pdf_directory):
    """Process all pdf files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all pdf files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"\nProcessing:{len(pdf_files)} PDF file to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing:{pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
                
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
        except Exception as e:
            print(f"  ✗ Error: {e}")
            
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")


Processing:3 PDF file to process

Processing:Data_analyst.pdf
  ✓ Loaded 1 pages

Processing:Maaz    Amir (1).pdf
  ✓ Loaded 2 pages

Processing:MAAZ_Amir_AI_Resume.pdf
  ✓ Loaded 1 pages

Total documents loaded: 4


In [6]:
all_pdf_documents

[Document(metadata={'producer': 'pdfTeX-1.40.29', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-10-29T17:43:22+00:00', 'author': '', 'keywords': '', 'moddate': '2026-10-29T17:43:22+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.29 (TeX Live 2026) kpathsea version 6.4.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\Data_analyst.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Data_analyst.pdf', 'file_type': 'pdf'}, page_content='Maaz Amir\n♂phone +91-9934327856 ✉ amirmaaz16@gmail.com /linkedin LinkedIn /github GitHub\nSummary\nComputer Science graduate with internship experience in backend development and database-driven applications. Skilled in\nPython, relational databases, structured data processing, and problem-solving, with experience building REST APIs and\nvalidating application data. Interested in applying analytical thinking and data-driven approaches to business metrics,\noperational report

In [6]:
### Text splitting and get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    '''Split Documents into smaller chunks for better Rag performance'''
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [8]:
chunks = split_documents(all_pdf_documents)
chunks

Split 4 documents into 16 chunks

Example chunk:
Content: Maaz Amir
♂phone +91-9934327856 ✉ amirmaaz16@gmail.com /linkedin LinkedIn /github GitHub
Summary
Computer Science graduate with internship experience in backend development and database-driven applica...
Metadata: {'producer': 'pdfTeX-1.40.29', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-10-29T17:43:22+00:00', 'author': '', 'keywords': '', 'moddate': '2026-10-29T17:43:22+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.29 (TeX Live 2026) kpathsea version 6.4.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\Data_analyst.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Data_analyst.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pdfTeX-1.40.29', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-10-29T17:43:22+00:00', 'author': '', 'keywords': '', 'moddate': '2026-10-29T17:43:22+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.29 (TeX Live 2026) kpathsea version 6.4.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\Data_analyst.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Data_analyst.pdf', 'file_type': 'pdf'}, page_content='Maaz Amir\n♂phone +91-9934327856 ✉ amirmaaz16@gmail.com /linkedin LinkedIn /github GitHub\nSummary\nComputer Science graduate with internship experience in backend development and database-driven applications. Skilled in\nPython, relational databases, structured data processing, and problem-solving, with experience building REST APIs and\nvalidating application data. Interested in applying analytical thinking and data-driven approaches to business metrics,\noperational report

### Embedding and VectorStore DB

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

d:\DEV Projects\RAG Projects\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
class EmbeddingManager:
     """Handles document embedding generation using SentenceTransformer"""
     
     def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()
        
     def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
        
     def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6356.22it/s]


Model loaded successfully. Embedding dimension: 384


### VectorStore

In [9]:
# from importlib import metadata


class VectorStore:
    '''Manages document embedding in a ChromaDB Vector Store'''
    
    def __init__(self,collection_name:str="pdf_documents",persist_directory:str="../data/vector_store"):
        '''
        Initialise the vector store
        
        Args:
            collection_name: Name of the ChromaDB Collection
            persist_directory: Directory to persist the vector store
        '''
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection=None
        self._initialize_store()
        
    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
        
    def add_documents(self,documents:List[Any],embeddings:np.ndarray):
        '''
        Add documents and thier embedding to the vector store
        
        Args:
            documents: List of Langchain documents
            embedding: Corresponding embeddings for the documents
        '''
        
        if len(documents) != len(embeddings):
            raise ValueError('Number of documents must match number of embeddings')
        
        print(f"Adding {len(documents)} document to the vector store ...")
        
        # Prepare data for the ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list=[]
        
        for i, (doc,embedding) in enumerate(zip(documents,embeddings)):
            # Generate unique id
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document Content
            documents_text.append(doc.page_content)
            
            #Embeddings
            embeddings_list.append(embeddings.tolist())
            
        # Add to collection
        try:
            if self.collection is None:
                raise RuntimeError("Vector store collection was not initialized")

            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
            

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0
